# 05 — DPO ile Alignment (Hizalama)

**Kapsam:** RLHF, DPO, PPO veya benzeri model hizalama (alignment) yöntemleri.

## RLHF/PPO vs. DPO
Klasik RLHF üç aşamalıdır: (1) SFT, (2) ayrı bir **ödül modeli** eğitmek, (3) bu ödül
modelini kullanarak **PPO** ile politika modelini güncellemek. Bu, iki model + kararsız
bir RL döngüsü gerektirir.

**DPO (Direct Preference Optimization)**, tercih çiftlerini (chosen/rejected) doğrudan
bir kayıp fonksiyonuna çevirir — ayrı ödül modeli veya RL rollout'u gerekmez. Matematiksel
olarak PPO'nun optimum noktasına eşdeğer bir çözüme ulaşır, ama çok daha kararlı ve
ucuzdur. Bu yüzden küçük ölçekli projelerde tercih ediyoruz.

Bu notebook, 04. notebook'ta ürettiğiniz QLoRA adaptörünün üzerine devam eder.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) unpack adımını atlar.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
#   4) EN ÖNEMLİSİ: data/, models/, mlruns/ klasörlerini Drive'daki kalıcı bir
#      klasöre sembolik bağlantı (symlink) yapar. Neden gerekli: /content her
#      runtime'da sıfırlanır, yani 01. notebook'ta ürettiğiniz corpus.jsonl gibi
#      dosyalar farklı bir runtime'da (örn. 02. notebook'u açtığınızda) KAYBOLUR.
#      Bu adım olmadan her notebook'u ayrı ayrı çalıştırdığınızda önceki adımların
#      ürettiği veriyi bulamazsınız. Sembolik bağlantı sayesinde hangi runtime'da
#      olursanız olun aynı kalıcı depoyu okur/yazarsınız.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"
DRIVE_DATA_DIR = "/content/drive/MyDrive/baykar-nlp-hazirlik-data"
PERSIST_DIRS = ["data", "models", "mlruns"]

try:
    from google.colab import drive
    # drive.mount() zaten mount edilmişse anında geri döner (idempotent);
    # os.path.exists("/content/drive") ile "mount edilmiş mi" kontrol etmek
    # güvenilmez çünkü klasör, başarısız/yarım bir mount denemesinden sonra
    # bile var olabilir. Bu yüzden koşulsuz çağırıyoruz.
    drive.mount("/content/drive", force_remount=True)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False  # Colab dışında (yerelde) çalışıyorsanız Drive adımları atlanır.

if not os.path.exists(PROJECT_DIR) and IN_COLAB:
    if os.path.exists(DRIVE_ZIP_PATH):
        import shutil
        shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
    else:
        print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
              "yükleyin ya da kendi reponuzu klonlayın: "
              f"!git clone <repo-url> {PROJECT_DIR}")

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

if IN_COLAB and os.path.exists(PROJECT_DIR):
    import shutil
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for _name in PERSIST_DIRS:
        _drive_path = os.path.join(DRIVE_DATA_DIR, _name)
        os.makedirs(_drive_path, exist_ok=True)
        _local_path = os.path.join(PROJECT_DIR, _name)

        if os.path.islink(_local_path):
            continue  # zaten Drive'a bağlanmış

        if os.path.isdir(_local_path):
            # Zip'ten gelen boş klasörü kaldırıp yerine symlink koyuyoruz. İçinde
            # (nadiren) veri varsa önce Drive'a taşıyoruz, hiçbir şeyi kaybetmiyoruz.
            for _item in os.listdir(_local_path):
                _src = os.path.join(_local_path, _item)
                _dst = os.path.join(_drive_path, _item)
                if not os.path.exists(_dst):
                    shutil.move(_src, _dst)
            shutil.rmtree(_local_path)

        os.symlink(_drive_path, _local_path)

    print("Kalıcı veri klasörü:", DRIVE_DATA_DIR)


## 1. Tercih (preference) veri seti üretimi

'Chosen' = iyi yapılandırılmış hedef doküman, 'rejected' = şablon bağlamı verilmeden üretilen serbest/tutarsız cevap.

In [ ]:
from src.alignment.preference_dataset import build_preference_dataset, save_preference_dataset

pairs = build_preference_dataset(max_examples=150)
path = save_preference_dataset(pairs)
print(f"{len(pairs)} tercih çifti kaydedildi -> {path}")
print("\nÖrnek:")
print("Chosen:", pairs[0]["chosen"][:200], "...")
print("Rejected:", pairs[0]["rejected"][:200], "...")


## 2. DPO eğitimi

In [ ]:
from src.alignment.dpo_train import train
from src.config import QLORA_CONFIG

dpo_adapter_path = train(sft_adapter_dir=QLORA_CONFIG.output_dir)
print("DPO adaptörü kaydedildi ->", dpo_adapter_path)


## 3. Karşılaştırma

Aynı taslak notları SFT-only ve SFT+DPO modelleriyle karşılaştırın; DPO sonrası
dokümanların daha tutarlı ve daha az 'uydurma ayrıntı içeren' olmasını bekleriz.

In [ ]:
from src.rag.rag_pipeline import answer

q = "Aşağıdaki kaba taslak notları iyi yapılandırılmış bir SSS bölümüne dönüştür.\n\nTaslak notlar:\n- şifre sıfırlama nasıl\n- veri saklama süresi ne kadar"
print("SFT-only:\n", answer(q, model_path=QLORA_CONFIG.output_dir)["answer"])
print("\nSFT+DPO:\n", answer(q, model_path=dpo_adapter_path)["answer"])
